In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import matplotlib.colors as mcolors
import seaborn as sns
import json
import re
import math
import numpy as np
from pathlib import Path

sns.set_theme(style="whitegrid")
plt.rcParams.update({'figure.autolayout': True, 'figure.dpi': 120})


# Global Constants
LANG_CPP = "C++ (EVE)"
LANG_PY = "Python (Scikit-Learn)"
PHASE_MAP = {
    "soa": "AoS to SoA",
    "pp": "K-Means++ Init",
    "lloyd": "Lloyd Algorithm"
}

# --- HELPER FUNCTIONS ---

def extract_config_params(filename):
    """Extracts Dimension, Samples, and Clusters from a string using regex."""
    match = re.search(r"(\d+)D_(\d+)S_(\d+)K", filename)
    if match:
        return int(match.group(1)), int(match.group(2)), int(match.group(3))
    return None, None, None

def create_subplot_grid(n_plots, cols=2, row_height=5, fig_width=12):
    """Handles the boilerplate for dynamic grid layouts and unused axes."""
    rows = math.ceil(n_plots / cols)
    fig, axes = plt.subplots(nrows=rows, ncols=cols, figsize=(fig_width, row_height * rows))
    
    axes = axes.flatten() if n_plots > 1 else np.array([axes])
    
    for j in range(n_plots, len(axes)):
        fig.delaxes(axes[j])
        
    return fig, axes

def extract_iterations_from_file(filepath):
    """Parses the Lloyd Iteration count from a results text file."""
    if not filepath.exists():
        return 1
    try:
        with open(filepath, 'r') as f:
            content = f.read()
            # Look for the tag and grab the number immediately following it
            match = re.search(r"\[Lloyd Iterations\]\s*\n\s*(\d+)", content)
            if match:
                return int(match.group(1))
    except Exception as e:
        print(f"Warning: Could not parse iterations from {filepath.name}: {e}")
    return 1

In [ ]:
def load_benchmark_data(data_dir="./datasets"):
    records = []
    path = Path(data_dir)
    
    for filepath in path.glob("*.json"):
        dim, samples, clusters = extract_config_params(filepath.name)
        if dim is None:
            print(f"Skipping malformed file: {filepath.name}")
            continue
            
        phase_key = filepath.name.split("_")[0]
        lang_key = filepath.name.split("_")[1]

        txt_filename = f"results_{lang_key}_{dim}D_{samples}S_{clusters}K.txt"
        txt_filepath = path / txt_filename
        
        iterations = extract_iterations_from_file(txt_filepath)
        
        with open(filepath, 'r') as f:
            data = json.load(f)
            
        for bench in data.get("benchmarks", []):
            for run in bench.get("runs", []):
                for val in run.get("values", []):
                    records.append({
                        "Phase": PHASE_MAP.get(phase_key, phase_key),
                        "Language": LANG_CPP if lang_key=="cpp" else LANG_PY,
                        "Dimensions": dim,
                        "Samples": samples,
                        "Clusters": clusters,
                        "Iterations": iterations if phase_key=="lloyd" else 1,
                        "Time_s": val,
                        "Configuration": f"{dim}D | {samples}S | {clusters}K"
                    })
                    
    df = pd.DataFrame(records)
    df["Phase"] = pd.Categorical(df["Phase"], categories=list(PHASE_MAP.values()), ordered=True)
    df["Language"] = pd.Categorical(df["Language"], categories=[LANG_CPP, LANG_PY], ordered=True)
    
    return df

df_bench = load_benchmark_data()

print(f"Loaded {len(df_bench)} individual iterations.")
df_bench.head()

In [ ]:
def compute_inertia(data_file, result_file, n_samples, n_features):
    """Reads the result file and calculates the sum of squared distances to centroids."""
    raw_data = np.fromfile(data_file, dtype=np.float32).reshape((n_samples, n_features))
    
    with open(result_file, 'r') as f:
        lines = f.read().splitlines()
        
    centroids = []
    clusters = {}
    
    mode = None
    for line in lines:
        if line == "[Centroids]":
            mode = "centroids"
            continue
        elif line == "[Clusters]":
            mode = "clusters"
            continue
            
        if mode == "centroids":
            centroids.append([float(x) for x in line.split()])
        elif mode == "clusters":
            parts = line.split(":")
            k = int(parts[0])
            indices_str = parts[1].strip()
            clusters[k] = [int(x) for x in indices_str.split()] if indices_str else []
            
    centroids = np.array(centroids, dtype=np.float32)
    inertia = 0.0
    
    for k, indices in clusters.items():
        if not indices:
            continue
        pts = raw_data[indices]
        c = centroids[k]
        inertia += np.sum((pts - c) ** 2)
        
    return inertia

def validate_lloyd_parity(data_dir="./datasets", tolerance_pct=0.01):
    path = Path(data_dir)
    records = []
    
    for cpp_file in path.glob("results_cpp_*.txt"):
        config_id = cpp_file.name.replace("results_cpp_", "").replace(".txt", "")
        py_file = path / f"results_py_{config_id}.txt"
        data_file = path / f"data_{config_id}.bin"
        
        if not py_file.exists() or not data_file.exists():
            continue
            
        dim, samples, clusters = extract_config_params(config_id)
        cpp_iters = extract_iterations_from_file(cpp_file)
        py_iters = extract_iterations_from_file(py_file)
        
        cpp_inertia = compute_inertia(data_file, cpp_file, samples, dim)
        py_inertia = compute_inertia(data_file, py_file, samples, dim)
        
        max_inertia = max(cpp_inertia, py_inertia)
        diff_pct = abs(cpp_inertia - py_inertia) / max_inertia * 100 if max_inertia > 0 else 0.0
        
        records.append({
            "Configuration": f"{dim}D | {samples}S | {clusters}K",
            "Diff (%)": diff_pct,
            "Status": "✅ PASS" if diff_pct <= tolerance_pct else "❌ FAIL",
            "Lloyd C++ Iteration": cpp_iters,
            "Lloyd Py Iterations": py_iters,
        })
        
    df_parity = pd.DataFrame(records)
    failures = df_parity[df_parity["Status"] == "❌ FAIL"]
    
    if not failures.empty:
        print(f"WARNING: Found {len(failures)} configurations outside {tolerance_pct}% limit!")
    else:
        print(f"SUCCESS: All {len(df_parity)} configurations match within limit!")
        
    return df_parity.sort_values(by="Diff (%)", ascending=False).reset_index(drop=True)

df_parity = validate_lloyd_parity(tolerance_pct=0.01)
display(df_parity[df_parity["Status"] == "❌ FAIL"])

In [ ]:
# Create a master Summary DataFrame to be used by all subsequent charts
df_lloyd = df_bench[df_bench["Phase"] == PHASE_MAP["lloyd"]]

df_mean = df_lloyd.groupby(
    ["Dimensions", "Samples", "Clusters", "Language"], observed=True
)["Time_s"].mean().reset_index()

df_summary = df_mean.pivot(
    index=["Dimensions", "Samples", "Clusters"], 
    columns="Language", 
    values="Time_s"
).reset_index()

df_summary["Speedup (x)"] = df_summary[LANG_PY] / df_summary[LANG_CPP]

In [ ]:
max_samples = df_bench["Samples"].max()

df_math_avg = df_bench[
    (df_bench["Phase"] == PHASE_MAP["lloyd"]) & 
    (df_bench["Language"] == LANG_CPP) & 
    (df_bench["Samples"] == max_samples)
].copy()
df_math_avg["Time_Per_Iter"] = df_math_avg["Time_s"] / df_math_avg["Iterations"]
df_math_summary = df_math_avg.groupby(["Clusters", "Dimensions"])["Time_Per_Iter"].mean().reset_index()

df_fixed_avg = df_bench[
    (df_bench["Phase"].isin([PHASE_MAP["soa"], PHASE_MAP["pp"]])) & 
    (df_bench["Language"] == LANG_CPP) & 
    (df_bench["Samples"] == max_samples)
].groupby(["Clusters", "Dimensions", "Phase"])["Time_s"].mean().reset_index()

df_plot = df_fixed_avg.merge(df_math_summary, on=["Clusters", "Dimensions"])
df_plot["Equivalent_Iters"] = df_plot["Time_s"] / df_plot["Time_Per_Iter"]
df_plot["Phase"] = df_plot["Phase"].cat.remove_unused_categories()

unique_clusters = sorted(df_plot["Clusters"].unique())
fig, axes = create_subplot_grid(len(unique_clusters), cols=2, row_height=5, fig_width=12)

for i, cluster in enumerate(unique_clusters):
    ax = axes[i]
    data_cluster = df_plot[df_plot["Clusters"] == cluster]
    
    sns.barplot(
        data=data_cluster,
        x="Dimensions",
        y="Equivalent_Iters",
        hue="Phase",
        ax=ax,
        palette={PHASE_MAP["soa"]: "#e74c3c", PHASE_MAP["pp"]: "#f39c12"}
    )
    
    ax.set_title(f"{cluster} Clusters")
    ax.set_ylabel("Cost (in equivalent Lloyd iterations)")
    ax.set_xlabel("Dimensions")
    ax.grid(axis='y', linestyle='--', alpha=0.6)

    if i != 0: 
        ax.get_legend().remove()
    
fig.suptitle(
    f"Architectural Overhead: Fixed Costs expressed in Lloyd Iterations\n"
    f"(How many iterations worth of time are spent on Setup/Memory? Samples: {df_bench['Samples'].max():,})",
    fontsize=16
)

plt.tight_layout()
plt.show()

In [ ]:
df_math = df_lloyd.copy()

df_math['Iteration'] = df_math.groupby(['Language', 'Dimensions', 'Samples', 'Clusters']).cumcount()

df_pivot = df_math.pivot(
    index=['Dimensions', 'Samples', 'Clusters', 'Iteration'], 
    columns='Language', 
    values='Time_s'
).reset_index()

df_pivot['Speedup (x)'] = df_pivot[LANG_PY] / df_pivot[LANG_CPP]
df_pivot['Cluster Group'] = df_pivot['Clusters'].astype(str) + " Clusters"

g = sns.relplot(
    data=df_pivot,
    x="Samples",
    y="Speedup (x)",
    hue="Cluster Group",    
    col="Dimensions",       
    col_wrap=2,             
    kind="scatter",
    alpha=0.6,     
    palette="Set1",         
    height=4,
    aspect=1.2,
    facet_kws={'sharey': False} 
)

g.set(xscale="log")
g.fig.suptitle("Absolute Speedup vs. Sample Size (Lloyd Iterations)", y=1.05)
g.set_axis_labels("Number of Samples", "Speedup Multiplier (x)")

for ax in g.axes.flat:
    ax.grid(True, linestyle='--', alpha=0.7)

plt.show()

df_pivot.drop(columns=['Cluster Group', 'Iteration'], inplace=True, errors='ignore')

In [ ]:
cluster_vals = sorted(df_summary["Clusters"].unique())
fig, axes = create_subplot_grid(len(cluster_vals), cols=2, row_height=6, fig_width=14)

for i, cluster in enumerate(cluster_vals):
    ax = axes[i]
    subset = df_summary[df_summary["Clusters"] == cluster]
    heatmap_pivot = subset.pivot(index="Dimensions", columns="Samples", values="Speedup (x)")
    
    sns.heatmap(
        heatmap_pivot, annot=True, fmt=".2f", cmap="turbo", 
        ax=ax, cbar=True, cbar_kws={'label': 'Speedup Multiplier (x)'}, linewidths=.5
    )
    
    ax.set_title(f"Clusters: {cluster}")
    ax.set_xlabel("Number of Samples")
    ax.set_ylabel("Dimensions" if i % 2 == 0 else "")
    ax.tick_params(axis='x', labelrotation=0)

plt.suptitle(f"Speedup Heatmap ({LANG_CPP} vs. {LANG_PY}) by Number of Clusters\nPhase: {PHASE_MAP["lloyd"]}", y=1.02, fontsize=16, fontweight='bold')
plt.show()

In [ ]:
# 1. Calculate the mean Python times to use as the static baseline
py_means = df_bench[df_bench["Language"] == LANG_PY].groupby(
    ["Phase", "Dimensions", "Samples", "Clusters"], observed=True
)["Time_s"].mean().reset_index().rename(columns={"Time_s": "Py_Mean_Time"})

# 2. Merge with raw C++ runs to preserve iteration variance for Seaborn's error bands
df_cpp = df_bench[df_bench["Language"] == LANG_CPP].copy()
df_norm = pd.merge(df_cpp, py_means, on=["Phase", "Dimensions", "Samples", "Clusters"])
df_norm["Speedup (x)"] = df_norm["Py_Mean_Time"] / df_norm["Time_s"]

# 3. Calculate baseline performance retention against the smallest cluster size (Base K)
base_k = df_norm["Clusters"].min()
baseline_df = df_norm[df_norm["Clusters"] == base_k].groupby(
    ["Phase", "Dimensions", "Samples"], observed=True
)["Speedup (x)"].mean().reset_index().rename(columns={"Speedup (x)": "Base_Speedup"})

df_norm = pd.merge(df_norm, baseline_df, on=["Phase", "Dimensions", "Samples"])
df_norm["Performance Retention (%)"] = (df_norm["Speedup (x)"] / df_norm["Base_Speedup"]) * 100

df_norm["Phase"] = df_norm["Phase"].cat.remove_unused_categories()

for phase in df_norm["Phase"].unique():
    phase_data = df_norm[df_norm["Phase"] == phase]
    
    g = sns.relplot(
        data=phase_data,
        x="Clusters",
        y="Performance Retention (%)",
        hue="Samples",   
        col="Dimensions",
        col_wrap=2,
        kind="line",
        errorbar="sd",
        marker="o",
        hue_norm=mcolors.LogNorm(),
        legend="full",
        palette="turbo",
        height=4,  
        aspect=1.2,
        facet_kws={'sharey': True}
    )

    g.fig.suptitle(
        f"PHASE: {phase.upper()}\nNormalized Efficiency against K={base_k}", 
        y=1.12, 
        fontsize=16, 
        fontweight='bold',
        bbox={'facecolor': 'lightgrey', 'alpha': 0.3, 'pad': 10}
    )
    
    g.set_axis_labels("Number of Clusters (K)", "Retention (%)")

    for ax in g.axes.flat:
        ax.axhline(100, color='red', linestyle='--', alpha=0.5)
        ax.grid(axis='y', linestyle='--', alpha=0.7)
        ax.xaxis.set_major_locator(mtick.MaxNLocator(integer=True))
        
    plt.show()

In [ ]:
median_val = min(df_bench["Clusters"].unique(), key=lambda x: abs(x - df_bench["Clusters"].median()))

df_line = df_bench[
    (df_bench["Clusters"] == median_val) & 
    (df_bench["Language"] == LANG_CPP) &
    (df_bench["Phase"] == PHASE_MAP["lloyd"])
].copy()

df_line["Phase"] = df_line["Phase"].cat.remove_unused_categories()

df_line["Time_per_Lloyd_Iter_per_Sample (ms)"] = df_line["Time_s"] / df_line["Samples"] / df_line["Iterations"] * 1000

g = sns.relplot(
    data=df_line,
    x="Samples",
    y="Time_per_Lloyd_Iter_per_Sample (ms)",
    col="Dimensions",
    col_wrap=3,      
    kind="line",
    errorbar="sd",
    marker="o",
    facet_kws={'sharey': False},
    height=4,
    aspect=1.2,
)

g.set(xscale="log") 

g.fig.suptitle(f"{LANG_CPP} Mean Time per Lloyd Iteration per Sample vs. Sample Size\n(Fixed at {median_val} Clusters)", y=1.1)
g.set_axis_labels("Number of Samples", "Time per Lloyd Iteration per Sample (ms)")

plt.show()